In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import getFamaFrenchFactors as gff
from sklearn.linear_model import LinearRegression

In [10]:
sp500_future = pd.read_excel('Sp_500_Futures.xlsx',index_col='Date')

In [17]:
# Futures Contract for SP 500
sp500_futures = pd.read_excel('Sp_500_Futures.xlsx',index_col='Date')
sp500_futures = sp500_futures.pct_change().dropna()
sp500_futures.rename(columns={'Adj Close':'Futures Pct Change'},inplace=True)
#sp500_futures.drop(sp500_futures.index[-1],inplace=True)
#sp500_futures.drop(sp500_futures.index[:13],inplace=True)

is_sp500_futures = sp500_futures.iloc[:-12]
is_sp500_futures.head()

,Futures Pct Change
Date,
2020-02-01,-0.084677
2020-04-01,-0.016435
2020-05-01,0.048062
2020-06-01,0.015861
2020-07-01,0.056063


In [18]:
os_sp500_futures = sp500_futures.iloc[-12:]
os_sp500_futures

,Futures Pct Change
Date,
2022-12-01,-0.053966
2023-02-01,0.029656
2023-03-01,0.040812
2023-04-01,0.012265
2023-05-01,0.000477
2023-06-01,0.071054
2023-07-01,0.028129
2023-08-01,-0.021346
2023-09-01,-0.042183


In [19]:
# Fama French Monthly Data using getFamaFrenchFactors module
ff3_monthly = gff.famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)
ff3_monthly.index = ff3_monthly.index.to_period('M').to_timestamp('D')
ff3_monthly = ff3_monthly.loc[is_sp500_futures.index]
ff3_monthly.head()

,Mkt-RF,SMB,HML,RF
Date,,,,
2020-02-01,-0.0813,0.0107,-0.0381,0.0012
2020-04-01,0.1365,0.0245,-0.0133,0.0000
2020-05-01,0.0558,0.0247,-0.0488,0.0001
2020-06-01,0.0246,0.0269,-0.0220,0.0001
2020-07-01,0.0577,-0.0233,-0.0141,0.0001


In [20]:
# Regression Analysis on Market Factor Vs SP 500 Futures Data 
y = ff3_monthly['Mkt-RF']
X = is_sp500_futures['Futures Pct Change'].values.reshape(-1,1)

model = LinearRegression()
model.fit(X, y)

print('Minimum variance Hedge ratio for Mkt-RF Factor:',model.coef_ )


Minimum variance Hedge ratio for Mkt-RF Factor: [0.82492161]


In [62]:
# Optimal Number of Contracts
size_of_position = 1000000
size_of_one_futures_contract = 250000
current_beta = model.coef_
target_beta = 0

# If current_beta > target_beta SHORT N number of contracts
N = (current_beta-target_beta) * (size_of_position)/(size_of_one_futures_contract)

# If target_beta > current_beta LONG N number of contracts
#N = (target_beta-current_beta) * (size_of_position)/(size_of_one_futures_contract)

N

array([3.29968644])

In [61]:
abs(N) * (sp500_future.loc['2023-02-01'] - sp500_future.loc['2023-12-01'])

Adj Close   -2786.5852
dtype: float64

In [60]:
sp500_future.tail(12)

,Adj Close
Date,
2022-12-01,3861.00
2023-02-01,3975.50
2023-03-01,4137.75
2023-04-01,4188.50
2023-05-01,4190.50
2023-06-01,4488.25
2023-07-01,4614.50
2023-08-01,4516.00
2023-09-01,4325.50


In [9]:
# Futures Contract for Russell 2000
russell2000_futures = pd.read_excel('Russell 2000 Futures Historical Data.xlsx',index_col='Date')
russell2000_futures = russell2000_futures.iloc[::-1].pct_change().dropna()
russell2000_futures.rename(columns={'Price':'Futures Pct Change'},inplace=True)
russell2000_futures


FileNotFoundError: [Errno 2] No such file or directory: 'Russell 2000 Futures Historical Data.xlsx'

In [ ]:
# Regression Analysis on Size Factor Vs Russell 2000 Futures Data 

y = ff3_monthly['SMB']
X = russell2000_futures['Futures Pct Change'].values.reshape(-1,1)

model = LinearRegression()
model.fit(X, y)

print('Minimum variance Hedge ratio for SMB Factor:',model.coef_ )

Minimum variance Hedge ratio for SMB Factor: [0.2289088]
